In [1]:
from google.colab import files

uploaded = files.upload()

Saving 경상남도 창원시_화물운송 주선업체 현황_20230516.csv to 경상남도 창원시_화물운송 주선업체 현황_20230516.csv


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

base_path = "/content/drive/MyDrive/contest_changwon_port"

raw_path = os.path.join(base_path, "data/raw")
processed_path = os.path.join(base_path, "data/processed")

os.makedirs(raw_path, exist_ok=True)
os.makedirs(processed_path, exist_ok=True)

print("base_path:", base_path)
print("raw_path:", raw_path)
print("processed_path:", processed_path)

Mounted at /content/drive
base_path: /content/drive/MyDrive/contest_changwon_port
raw_path: /content/drive/MyDrive/contest_changwon_port/data/raw
processed_path: /content/drive/MyDrive/contest_changwon_port/data/processed


In [4]:
forwarder_folder = os.path.join(base_path, "data/raw/freight_forwarder")
os.makedirs(forwarder_folder, exist_ok=True)

print("주선업체 폴더:", forwarder_folder)

주선업체 폴더: /content/drive/MyDrive/contest_changwon_port/data/raw/freight_forwarder


In [5]:
import glob
import os

files = glob.glob(os.path.join(forwarder_folder, "*.csv"))

print("파일 수:", len(files))

for f in files:
    print(os.path.basename(f))

파일 수: 1
경상남도 창원시_화물운송 주선업체 현황_20230516.csv


In [6]:
import pandas as pd

forwarder_path = os.path.join(
    forwarder_folder,
    "경상남도 창원시_화물운송 주선업체 현황_20230516.csv"
)

forwarder = pd.read_csv(forwarder_path, encoding="cp949")

print("데이터 크기:", forwarder.shape)
display(forwarder.head())
print(forwarder.columns)

데이터 크기: (76, 6)


,해당구,연번,업체명,대표자,주소,비고
0,의창구,1,KGB창원합포점,이미숙,경상남도 창원시 의창구 금강로355번길 35 (소계동),NaN
1,의창구,2,국민포장이사,김세연,경상남도 창원시 의창구 남산로9번길 54-6 (팔용동),NaN
2,의창구,3,개미익스프레스,박중규,경상남도 창원시 의창구 남산로9번길 54-6 (팔용동),NaN
3,의창구,4,창원이사 경남익스프레스,안군상,경상남도 창원시 의창구 남산로9번길 54-6 (팔용동),NaN
4,의창구,5,삼성익스프레스,이보영,경상남도 창원시 의창구 남산로9번길 54-6 (팔용동),NaN


Index(['해당구', '연번', '업체명', '대표자', '주소', '비고'], dtype='object')


In [7]:
# 구별 주선업체 수

forwarder_gu_summary = (
    forwarder
    .groupby("해당구", as_index=False)
    .agg(
        forwarder_count=("업체명", "count")
    )
    .sort_values("forwarder_count", ascending=False)
)

display(forwarder_gu_summary)

,해당구,forwarder_count
3,의창구,27
1,마산회원구,24
0,마산합포구,9
4,진해구,9
2,성산구,7


In [8]:
total_forwarder = forwarder_gu_summary["forwarder_count"].sum()

forwarder_gu_summary["ratio"] = (
    forwarder_gu_summary["forwarder_count"] / total_forwarder * 100
)

display(forwarder_gu_summary)

,해당구,forwarder_count,ratio
3,의창구,27,35.526316
1,마산회원구,24,31.578947
0,마산합포구,9,11.842105
4,진해구,9,11.842105
2,성산구,7,9.210526


In [9]:
# 항만 인접지역 / 내륙지역 구분

def classify_area(gu):
    if gu in ["마산합포구", "진해구"]:
        return "항만 인접지역"
    else:
        return "내륙·산단 배후지역"

forwarder["area_type"] = forwarder["해당구"].apply(classify_area)

forwarder_area_summary = (
    forwarder
    .groupby("area_type", as_index=False)
    .agg(
        forwarder_count=("업체명", "count")
    )
)

forwarder_area_summary["ratio"] = (
    forwarder_area_summary["forwarder_count"] /
    forwarder_area_summary["forwarder_count"].sum() * 100
)

display(forwarder_area_summary)

,area_type,forwarder_count,ratio
0,내륙·산단 배후지역,58,76.315789
1,항만 인접지역,18,23.684211


In [10]:
forwarder["dong"] = forwarder["주소"].astype(str).str.extract(r"\(([^)]+)\)")[0]

display(forwarder[["해당구", "업체명", "주소", "dong"]].head())

,해당구,업체명,주소,dong
0,의창구,KGB창원합포점,경상남도 창원시 의창구 금강로355번길 35 (소계동),소계동
1,의창구,국민포장이사,경상남도 창원시 의창구 남산로9번길 54-6 (팔용동),팔용동
2,의창구,개미익스프레스,경상남도 창원시 의창구 남산로9번길 54-6 (팔용동),팔용동
3,의창구,창원이사 경남익스프레스,경상남도 창원시 의창구 남산로9번길 54-6 (팔용동),팔용동
4,의창구,삼성익스프레스,경상남도 창원시 의창구 남산로9번길 54-6 (팔용동),팔용동


In [11]:
# 동별 주선업체 수

forwarder_dong_summary = (
    forwarder
    .groupby(["해당구", "dong"], as_index=False)
    .agg(
        forwarder_count=("업체명", "count")
    )
    .sort_values("forwarder_count", ascending=False)
)

display(forwarder_dong_summary.head(20))

,해당구,dong,forwarder_count
28,의창구,팔용동,8
12,마산회원구,회원동,7
22,의창구,봉곡동,6
6,마산회원구,구암동,6
3,마산합포구,상남동,4
7,마산회원구,석전동,4
11,마산회원구,회성동,3
16,성산구,신월동,2
33,진해구,석동,2
21,의창구,명서동,2


In [12]:
processed_path = os.path.join(base_path, "data/processed")
os.makedirs(processed_path, exist_ok=True)

forwarder_gu_summary.to_csv(
    os.path.join(processed_path, "freight_forwarder_gu_summary_2023.csv"),
    index=False,
    encoding="utf-8-sig"
)

forwarder_area_summary.to_csv(
    os.path.join(processed_path, "freight_forwarder_area_summary_2023.csv"),
    index=False,
    encoding="utf-8-sig"
)

forwarder_dong_summary.to_csv(
    os.path.join(processed_path, "freight_forwarder_dong_summary_2023.csv"),
    index=False,
    encoding="utf-8-sig"
)

forwarder.to_csv(
    os.path.join(processed_path, "freight_forwarder_processed_2023.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("화물운송 주선업체 현황 분석 결과 저장 완료")

화물운송 주선업체 현황 분석 결과 저장 완료


화물운송 주선업체 현황 분석 결과, 창원시 내 주선업체는 총 76개로 확인되었으며, 의창구 27개, 마산회원구 24개, 마산합포구 9개, 진해구 9개, 성산구 7개 순으로 나타났다. 항만 인접지역인 마산합포구와 진해구에는 총 18개 업체가 분포하여 전체의 약 23.7%를 차지하였고, 의창구·성산구·마산회원구 등 내륙 및 산업 배후지역에는 58개 업체가 분포하여 약 76.3%를 차지하였다. 이는 창원시의 화물 배차·중개 기능이 항만 인접지역보다 내륙 및 산업 배후지역에 더 많이 분포하고 있음을 보여줌